In [1]:
import sys
from pathlib import Path
import os 

os.chdir("/Users/link/gpt-workout")

In [2]:
import torch 
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("./tokenizer/code_bpe_16k")
print("Vocab size:", len(tokenizer))
print("EOS token ID:", tokenizer.eos_token_id)

sample = "def hello():\n    print('안녕! 🐍')"

ids = tokenizer.encode(sample)
print("Tokens:", tokenizer.convert_ids_to_tokens(ids))
print("Decoded:", tokenizer.decode(ids))

assert tokenizer.decode(ids) == sample

# train, val = create_dataloaders('./data/dataset_generate/train.txt','./data/dataset_generate/val.txt',tokenizer, block_size, batch_size)

/Users/link/miniconda3/envs/transformer/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vocab size: 16384
EOS token ID: 0
Tokens: ['def', 'Ġhe', 'llo', '():', 'Ċ', 'ĠĠĠ', 'Ġprint', "('", 'ì', 'ķ', 'Ī', 'ë', 'ħ', 'ķ', '!', 'Ġ', 'ðŁ', 'Ĳ', 'į', "')"]
Decoded: def hello():
    print('안녕! 🐍')


단어 사이즈는 16384 로 고정시킨다. 

In [3]:
from data.data import create_dataloaders

train, val = create_dataloaders('./data/dataset_generate/train.txt','./data/dataset_generate/val.txt',tokenizer, block_size=512, batch_size=16) 

Loading cached tokens: data/dataset_generate/train.ce8b9ee4fe57.bin
Loading cached tokens: data/dataset_generate/val.96ec646f16cb.bin


In [4]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [5]:
from config import Config 
from gpt import GPT


cfg = Config(vocab_size=len(tokenizer), block_size=512, n_layer=16, n_embd=768, n_head=12, dropout=0.1)
model = GPT(cfg).to(device) 

total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(f"Model size: {total_params / 1e6:.2f}M")

learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) 

print(f"learnable parameters: {learnable_params:,}")
print(f"Model size: {learnable_params / 1e6:.2f}M")

Total parameters: 138,982,912
Model size: 138.98M
learnable parameters: 138,982,912
Model size: 138.98M


In [7]:
from helper import print_memory 

print_memory()

RAM 사용량: 0.59 GB


In [12]:
from optimizer import build_optimizer, set_learning_rate
from config import TrainConfig

train_cfg = TrainConfig()

use_amp = True

scaler = torch.amp.GradScaler(
    "mps",
    enabled=use_amp
)

optimizer = build_optimizer(model, train_cfg)

In [8]:
@torch.no_grad()
def evaluate(model, val_loader, max_batches=20):
    model.eval()
    losses = []

    for i, (x, y) in enumerate(val_loader):
        if i >= max_batches:
            break

        x, y = x.to("mps"), y.to("mps")

        with torch.autocast("mps", dtype=torch.float16):
            _, loss = model(x, y)

        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)

evaluate(model, train)

9.83477463722229

In [ ]:
from optimizer import set_learning_rate, learning_rate_at, build_optimizer
from config import TrainConfig

epochs = 3
warmup_steps = 500
eval_every = 200
log_every = 50

total_steps = epochs * len(train)
step = 0

train_cfg = TrainConfig(max_iters=total_steps, batch_size=16)
optimizer = build_optimizer(model, train_cfg)

model.train()
scaler = torch.amp.GradScaler("mps")


for epoch in range(epochs):
    for x, y in train:
        x, y = x.to("mps"), y.to("mps")

        # Learning rate
        set_learning_rate(optimizer, step, train_cfg)
        # Forward
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("mps", dtype=torch.float16):
            logits, loss = model(x, y)

        # Backward
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), train_cfg.grad_clip
        )

        scaler.step(optimizer)
        scaler.update()

        step += 1

        lr = learning_rate_at(step, train_cfg)
        # Logging
        if step % log_every == 0:
            print(f"step {step} | train loss {loss.item():.4f} | lr {lr:.2e}")

        # Validation
        if step % eval_every == 0:
            val_loss = evaluate(model, val)
            print(f"step {step} | val loss {val_loss:.4f}")